In [40]:
import os
import random
import collections
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt

# Scikit-learn Preprocessing, Splitting, and Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score, matthews_corrcoef, precision_recall_curve, auc, brier_score_loss
from sklearn.calibration import calibration_curve, CalibrationDisplay

# Imbalanced Learning Frameworks
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

# Machine Learning & Deep Learning Frameworks
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Explainable AI (XAI) Libraries
import shap
import lime
import lime.lime_tabular

import preprocess

In [41]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Ensure fully deterministic behavior in PyTorch backends
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [42]:
filepath = "data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
df = preprocess.load_and_clean_dataset(filepath)

#change to a binary label
df['Label'] = df['Label'].astype(str).str.strip().str.upper()
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

X = df.drop(columns=['Label'])
y = df['Label'].values

print(f"[*] Cleaned Feature matrix shape: {X.shape}")
print(f"[*] Target distribution: Benign (0) = {np.sum(y == 0)}, Attack (1) = {np.sum(y == 1)}")


[*] Loading raw dataset from: data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv


c:\Users\Moritz\Documents\Uni\bachlor_thesis\ids-xai-thesis\preprocess.py:12: DtypeWarning: Columns (0: Flow ID, 1:  Source IP, 2:  Destination IP, 3:  Timestamp, 4:  Label) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='latin-1')


[-] Dropped identifier columns: ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Timestamp']
[!] Purged 288737 malformed rows containing Infinity or empty cells.
[*] Cleaned Feature matrix shape: (170231, 78)
[*] Target distribution: Benign (0) = 168051, Attack (1) = 2180


In [43]:
# Step A: Split off 30% of the data into a temporary block, stratifying to preserve class ratios
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Step B: Split the temporary block evenly to yield a 15% Validation set and a 15% Test set
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("==================== DATA SPLIT SUMMARY ====================")
print(f"Training Set (70%):   X = {X_train.shape}, y = {y_train.shape}")
print(f"Validation Set (15%): X = {X_val.shape}, y = {y_val.shape}")
print(f"Test Set (15%):       X = {X_test.shape}, y = {y_test.shape}")

==================== DATA SPLIT SUMMARY ====================
Training Set (70%):   X = (119161, 78), y = (119161,)
Validation Set (15%): X = (25535, 78), y = (25535,)
Test Set (15%):       X = (25535, 78), y = (25535,)


In [44]:
# 1. Fit scaler ONLY on training data
scaler = MinMaxScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# 2. Correlated Feature Groups (Supervisor point 15 & 16)
corr_matrix = X_train_scaled.corr(method="spearman")
# Get upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# Find columns with correlation > 0.90
to_drop = [column for column in upper.columns if any(upper[column].abs() > 0.90)]

print(f"[*] Features identified for potential removal (>0.90 correlation): {len(to_drop)}")

[*] Features identified for potential removal (>0.90 correlation): 39


In [45]:
# Initialize SMOTE-Tomek with a fixed seed for strict reproducibility
smote_tomek = SMOTETomek(random_state=42)

# Resample ONLY the training set
X_train_resampled, y_train_resampled = smote_tomek.fit_resample(
    X_train_scaled,
    y_train
)

print("================= RESAMPLING SUMMARY =================")
print(f"Original Training Class Ratios: 0 = {np.sum(y_train == 0)}, 1 = {np.sum(y_train == 1)}")
print(f"Resampled Training Data Shape:  X = {X_train_resampled.shape}, y = {y_train_resampled.shape}")
print(f"Resampled Training Class Ratios: 0 = {np.sum(y_train_resampled == 0)}, 1 = {np.sum(y_train_resampled == 1)}")

================= RESAMPLING SUMMARY =================
Original Training Class Ratios: 0 = 117635, 1 = 1526
Resampled Training Data Shape:  X = (235254, 78), y = (235254,)
Resampled Training Class Ratios: 0 = 117627, 1 = 117627


In [46]:
print("[*] Initializing XGBoost Classifier...")

# Initialize XGBoost with strict reproducibility parameters
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=0.1,
    random_state=42,
    use_label_encoder=False,
    early_stopping_rounds=15
)

# Fit model with early stopping monitored against the validation split
xgb_model.fit(
    X_train_resampled, 
    y_train_resampled,
    eval_set=[(X_val_scaled, y_val)],
    verbose=False
)

print(f"[*] XGBoost training complete.")
print(f"    -> Best Iteration: {xgb_model.best_iteration}")

[*] Initializing XGBoost Classifier...


c:\Users\Moritz\Documents\Uni\bachlor_thesis\ids-xai-thesis\thesis_env\Lib\site-packages\xgboost\callback.py:385: UserWarning: [16:50:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[*] XGBoost training complete.
    -> Best Iteration: 398


In [47]:
class RobustNetworkSecurityDNN(nn.Module):
    def __init__(self, input_dim):
        super(RobustNetworkSecurityDNN, self).__init__()
        
        # Layer 1: Input to Hidden 1
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.30)
        
        # Layer 2: Hidden 1 to Hidden 2
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.30)
        
        # Layer 3: Hidden 2 to Output Layer
        self.fc3 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        x = self.sigmoid(self.fc3(x))
        return x

# Set calculation device backend
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_features_count = X_train_resampled.shape[1]

# Instantiate model architecture
model_dnn = RobustNetworkSecurityDNN(input_dim=input_features_count).to(device)
print(f"[*] PyTorch Network mapped successfully onto target hardware device: {device.type.upper()}")

[*] PyTorch Network mapped successfully onto target hardware device: CPU


In [48]:
# Convert DataFrames/Arrays to PyTorch multi-dimensional tensors
train_dataset = TensorDataset(
    torch.FloatTensor(X_train_resampled.values),
    torch.FloatTensor(y_train_resampled).unsqueeze(1)
)
val_x_tensor = torch.FloatTensor(X_val_scaled.values).to(device)
val_y_tensor = torch.FloatTensor(y_val).unsqueeze(1).to(device)

# Configure data loader iterations
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

# Set optimizer equations and standard binary loss scoring
criterion = nn.BCELoss()
optimizer = optim.Adam(model_dnn.parameters(), lr=0.001)

# Early Stopping parameters
patience = 10
best_val_loss = float('inf')
best_model_weights = None
patience_counter = 0
max_epochs = 150

print("[*] Initiating DNN optimization loop...")
for epoch in range(1, max_epochs + 1):
    model_dnn.train()
    running_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model_dnn(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        
    # Validation Evaluation Phase (Zero Leakage Check)
    model_dnn.eval()
    with torch.no_grad():
        val_outputs = model_dnn(val_x_tensor)
        val_loss = criterion(val_outputs, val_y_tensor).item()
        
    epoch_train_loss = running_loss / len(train_dataset)
    
    # Early stopping criteria tracking
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = model_dnn.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print(f"[*] Early stopping triggered at Epoch {epoch}. Overfitting threshold neutralized.")
        break

# Roll back parameter matrix states to the optimal captured validation validation loss weights
if best_model_weights is not None:
    model_dnn.load_state_dict(best_model_weights)
print(f"[*] Restored optimal model configurations. Best Validation Loss: {best_val_loss:.5f}")

[*] Initiating DNN optimization loop...
[*] Early stopping triggered at Epoch 21. Overfitting threshold neutralized.
[*] Restored optimal model configurations. Best Validation Loss: 0.04361


In [49]:
# Helper function to generate clean inference probability matrices from our DNN
def get_dnn_probabilities(df_input):
    model_dnn.eval()
    with torch.no_grad():
        tensor_input = torch.FloatTensor(df_input.values).to(device)
        probs = model_dnn(tensor_input).cpu().numpy().flatten()
    return probs

# Step A: Collect raw feature inference array probabilities from validation subsets
xgb_val_probs = xgb_model.predict_proba(X_val_scaled)[:, 1]
dnn_val_probs = get_dnn_probabilities(X_val_scaled)

# Step B: Declare target tuning thresholds range
thresholds_pool = np.arange(0.50, 0.96, 0.05)

best_xgb_threshold = 0.50
best_xgb_f1 = 0.0
best_dnn_threshold = 0.50
best_dnn_f1 = 0.0

print("================= VAL THRESHOLD CALIBRATION =================")
for t in thresholds_pool:
    # Evaluate XGBoost arrays
    xgb_preds = (xgb_val_probs >= t).astype(int)
    xgb_f1 = f1_score(y_val, xgb_preds, zero_division=0)
    if xgb_f1 > best_xgb_f1:
        best_xgb_f1 = xgb_f1
        best_xgb_threshold = t
        
    # Evaluate DNN arrays
    dnn_preds = (dnn_val_probs >= t).astype(int)
    dnn_f1 = f1_score(y_val, dnn_preds, zero_division=0)
    if dnn_f1 > best_dnn_f1:
        best_dnn_f1 = dnn_f1
        best_dnn_threshold = t
        
    print(f"Threshold: {t:.2f} | XGB F1: {xgb_f1:.4f} | DNN F1: {dnn_f1:.4f}")

print("\n[*] Calibrated Decision Parameter Options Frozen:")
print(f"    -> Selected Frozen XGBoost Threshold: {best_xgb_threshold:.2f} (Val F1: {best_xgb_f1:.4f})")
print(f"    -> Selected Frozen DNN Threshold:     {best_dnn_threshold:.2f} (Val F1: {best_dnn_f1:.4f})")

================= VAL THRESHOLD CALIBRATION =================
Threshold: 0.50 | XGB F1: 0.9954 | DNN F1: 0.5496
Threshold: 0.55 | XGB F1: 0.9954 | DNN F1: 0.5595
Threshold: 0.60 | XGB F1: 0.9954 | DNN F1: 0.5692
Threshold: 0.65 | XGB F1: 0.9954 | DNN F1: 0.5834
Threshold: 0.70 | XGB F1: 0.9954 | DNN F1: 0.6044
Threshold: 0.75 | XGB F1: 0.9970 | DNN F1: 0.6211
Threshold: 0.80 | XGB F1: 0.9970 | DNN F1: 0.6349
Threshold: 0.85 | XGB F1: 0.9969 | DNN F1: 0.8587
Threshold: 0.90 | XGB F1: 0.9969 | DNN F1: 0.8575
Threshold: 0.95 | XGB F1: 0.9985 | DNN F1: 0.8584

[*] Calibrated Decision Parameter Options Frozen:
    -> Selected Frozen XGBoost Threshold: 0.95 (Val F1: 0.9985)
    -> Selected Frozen DNN Threshold:     0.85 (Val F1: 0.8587)


In [50]:
# Step A: Extract test predictions using frozen configurations
xgb_test_probs = xgb_model.predict_proba(X_test_scaled)[:, 1]
dnn_test_probs = get_dnn_probabilities(X_test_scaled)

xgb_test_preds = (xgb_test_probs >= best_xgb_threshold).astype(int)
dnn_test_preds = (dnn_test_probs >= best_dnn_threshold).astype(int)

# Step B: Print formal validation summaries for your thesis report
print("=================== FINAL FROZEN XGBOOST REPORT ===================")
print(f"Applied Decision Threshold: {best_xgb_threshold:.2f}")
print(confusion_matrix(y_test, xgb_test_preds))
print(classification_report(y_test, xgb_test_preds, digits=4))

print("\n==================== FINAL FROZEN DNN REPORT ====================")
print(f"Applied Decision Threshold: {best_dnn_threshold:.2f}")
print(confusion_matrix(y_test, dnn_test_preds))
print(classification_report(y_test, dnn_test_preds, digits=4))

=================== FINAL FROZEN XGBOOST REPORT ===================
Applied Decision Threshold: 0.95
[[25207     1]
 [    4   323]]
              precision    recall  f1-score   support

           0     0.9998    1.0000    0.9999     25208
           1     0.9969    0.9878    0.9923       327

    accuracy                         0.9998     25535
   macro avg     0.9984    0.9939    0.9961     25535
weighted avg     0.9998    0.9998    0.9998     25535


==================== FINAL FROZEN DNN REPORT ====================
Applied Decision Threshold: 0.85
[[25111    97]
 [   18   309]]
              precision    recall  f1-score   support

           0     0.9993    0.9962    0.9977     25208
           1     0.7611    0.9450    0.8431       327

    accuracy                         0.9955     25535
   macro avg     0.8802    0.9706    0.9204     25535
weighted avg     0.9962    0.9955    0.9957     25535



In [51]:
# Standardized probability wrappers for the explainers
def xgb_predict_proba_wrapper(x_numpy):
    # Map numpy matrix rows directly back to pandas format to preserve column naming indices natively
    df_temp = pd.DataFrame(x_numpy, columns=X_train_scaled.columns)
    return xgb_model.predict_proba(df_temp)

def dnn_predict_proba_wrapper(x_numpy):
    df_temp = pd.DataFrame(x_numpy, columns=X_train_scaled.columns)
    probs_class_1 = get_dnn_probabilities(df_temp)
    probs_class_0 = 1.0 - probs_class_1
    return np.column_stack((probs_class_0, probs_class_1))

# step 6: correlation analysis

## phase 1 correlation grouping

In [52]:
def compute_correlation_groups(df_train, threshold=0.90):
    """
    Computes Spearman correlation matrix and groups features into clusters
    using connected components analysis based on a correlation threshold.
    """
    print(f"[*] Calculating pairwise Spearman correlation matrix for {df_train.shape[1]} features...")
    # Compute the absolute Spearman rank correlation matrix
    corr_matrix = df_train.corr(method='spearman').abs()
    
    # Create an adjacency matrix where 1 indicates correlation >= threshold
    adj_matrix = (corr_matrix >= threshold).astype(int)
    
    # Find connected components in the correlation graph
    n_components, labels = connected_components(adj_matrix.values, directed=False)
    
    # Map component labels back to feature names
    feature_names = df_train.columns
    group_to_features = collections.defaultdict(list)
    for idx, label in enumerate(labels):
        group_to_features[label].append(feature_names[idx])
        
    # Build maps and designate a deterministic representative feature for each cluster
    feature_to_group = {}
    final_groups = {}
    representative_features = {}
    
    group_counter = 0
    independent_counter = 0
    
    # Sort groups by size to process larger clusters first
    sorted_groups = sorted(group_to_features.values(), key=len, reverse=True)
    
    for features in sorted_groups:
        if len(features) > 1:
            # Multi-feature cluster
            group_name = f"Group_{group_counter}"
            group_counter += 1
        else:
            # Singleton independent feature
            group_name = f"Independent_{independent_counter}"
            independent_counter += 1
            
        final_groups[group_name] = sorted(features)
        
        # Select representative feature deterministically (first alphabetically)
        rep_feat = sorted(features)[0]
        representative_features[group_name] = rep_feat
        
        for feat in features:
            feature_to_group[feat] = group_name
            
    print(f"[+] Discovered {group_counter} highly correlated feature groups and {independent_counter} independent features.")
    return final_groups, feature_to_group, representative_features

# Run the correlation grouping pipeline on the training set
CORRELATION_THRESHOLD = 0.90
group_to_features, feature_to_group, group_representatives = compute_correlation_groups(
    X_train_scaled, 
    threshold=CORRELATION_THRESHOLD
)

# =====================================================================
# ACADEMIC LOGGING & DICTIONARY VERIFICATION
# =====================================================================
print("\n" + "="*80)
print(f"   SUMMARY OF DISCOVERED HIGHLY CORRELATED FEATURE GROUPS (Threshold >= {CORRELATION_THRESHOLD})")
print("="*80)

total_grouped_features = 0
for g_name, features in group_to_features.items():
    if "Group_" in g_name:
        total_grouped_features += len(features)
        print(f"-> {g_name} ({len(features)} features) | Rep: '{group_representatives[g_name]}'")
        for feat in features:
            print(f"   - {feat}")
        print("-"*80)

print(f"[*] Analysis complete. Total features absorbed into multi-variable groups: {total_grouped_features}")
print(f"[*] Remaining standalone independent features: {len(X_train_scaled.columns) - total_grouped_features}")
print("="*80)

[*] Calculating pairwise Spearman correlation matrix for 78 features...
[+] Discovered 13 highly correlated feature groups and 26 independent features.

   SUMMARY OF DISCOVERED HIGHLY CORRELATED FEATURE GROUPS (Threshold >= 0.9)
-> Group_0 (10 features) | Rep: 'Average Packet Size'
   - Average Packet Size
   - Avg Bwd Segment Size
   - Bwd Packet Length Max
   - Bwd Packet Length Mean
   - Max Packet Length
   - Packet Length Mean
   - Packet Length Std
   - Packet Length Variance
   - Subflow Bwd Bytes
   - Total Length of Bwd Packets
--------------------------------------------------------------------------------
-> Group_1 (8 features) | Rep: 'Fwd Header Length'
   - Fwd Header Length
   - Fwd Header Length.1
   - Fwd IAT Max
   - Fwd IAT Mean
   - Fwd IAT Total
   - Subflow Fwd Packets
   - Total Fwd Packets
   - act_data_pkt_fwd
--------------------------------------------------------------------------------
-> Group_2 (6 features) | Rep: 'Bwd Header Length'
   - Bwd Header Leng

In [54]:


# ---------------------------------------------------------------------
# 1. CORE DEPENDENCIES & UTILITIES (Natively defined to prevent missing variables)
# ---------------------------------------------------------------------
feature_list = X_train_scaled.columns.tolist()
num_features_original = len(feature_list)
all_feature_pool = np.arange(num_features_original)

# Initialize deterministic random number generator if not defined globally
if 'rng' not in globals():
    rng = np.random.default_rng(seed=42)

def extract_aligned_lime_attributions(instance, predict_fn, num_features):
    """Programmatic LIME feature alignment using feature indices."""
    exp = lime_explainer.explain_instance(
        data_row=instance,
        predict_fn=predict_fn,
        num_features=num_features
    )
    lime_vector = np.zeros(num_features)
    for feat_idx, weight in exp.local_exp[1]:
        lime_vector[feat_idx] = weight
    return lime_vector, exp.score

def extract_aligned_shap_attributions(instance, explainer):
    """Programmatic SHAP extraction supporting modern 3D array dimensions."""
    shap_vals = explainer.shap_values(instance.reshape(1, -1))
    if isinstance(shap_vals, list):
        # Handle dual class output list structure
        shap_vector = shap_vals[1].flatten()
    elif isinstance(shap_vals, np.ndarray):
        if len(shap_vals.shape) == 3: # (samples, features, classes)
            shap_vector = shap_vals[0, :, 1]
        elif len(shap_vals.shape) == 2:
            shap_vector = shap_vals.flatten()
        else:
            shap_vector = shap_vals.flatten()
    else:
        shap_vector = np.array(shap_vals).flatten()
    return shap_vector

def compute_bootstrapped_ci(data_series, n_bootstrap=5000, ci=0.95):
    """Computes academic mean along with empirical percentile intervals."""
    clean_data = data_series.dropna().values
    if len(clean_data) == 0:
        return 0.0, 0.0, 0.0
    boot_means = []
    for _ in range(n_bootstrap):
        sample = rng.choice(clean_data, size=len(clean_data), replace=True)
        boot_means.append(np.mean(sample))
    
    mean_val = np.mean(clean_data)
    lower_bound = np.percentile(boot_means, ((1.0 - ci) / 2.0) * 100)
    upper_bound = np.percentile(boot_means, (ci + (1.0 - ci) / 2.0) * 100)
    return mean_val, lower_bound, upper_bound

# ---------------------------------------------------------------------
# 2. STRATIFIED COHORT SAMPLING VERIFICATION
# ---------------------------------------------------------------------
# Safe check: if cohort_samples wasn't copied from Notebook 1, re-generate it on the fly
if 'cohort_samples' not in globals():
    print("[!] 'cohort_samples' not found in local kernel namespace. Re-generating cohorts programmatically...")
    
    # Get model probabilities and predictions to reconstruct the 250 evaluation sample spaces
    xgb_probs = xgb_model.predict_proba(X_test_scaled)[:, 1]
    xgb_preds = (xgb_probs >= best_xgb_threshold).astype(int)
    
    dnn_probs = get_dnn_probabilities(X_test_scaled)
    dnn_preds = (dnn_probs >= best_dnn_threshold).astype(int)
    
    # Stratify into standard target auditing categories
    tp_indices = np.where((y_test == 1) & (xgb_preds == 1) & (dnn_preds == 1))[0]
    tn_indices = np.where((y_test == 0) & (xgb_preds == 0) & (dnn_preds == 0))[0]
    fp_indices = np.where((y_test == 0) & ((xgb_preds == 1) | (dnn_preds == 1)))[0]
    fn_indices = np.where((y_test == 1) & ((xgb_preds == 0) | (dnn_preds == 0)))[0]
    
    # Borderline cases close to thresholds
    xgb_margin = np.abs(xgb_probs - best_xgb_threshold)
    dnn_margin = np.abs(dnn_probs - best_dnn_threshold)
    borderline_indices = np.argsort(xgb_margin + dnn_margin)[:100]
    
    # Deterministic sampling from each available target stack (50 rows per segment)
    cohort_samples = {
        'True Positives': rng.choice(tp_indices, min(50, len(tp_indices)), replace=False).tolist() if len(tp_indices) > 0 else [],
        'True Negatives': rng.choice(tn_indices, min(50, len(tn_indices)), replace=False).tolist() if len(tn_indices) > 0 else [],
        'False Positives': rng.choice(fp_indices, min(50, len(fp_indices)), replace=False).tolist() if len(fp_indices) > 0 else [],
        'False Negatives': rng.choice(fn_indices, min(50, len(fn_indices)), replace=False).tolist() if len(fn_indices) > 0 else [],
        'Borderline Cases': rng.choice(borderline_indices, min(50, len(borderline_indices)), replace=False).tolist() if len(borderline_indices) > 0 else []
    }

# ---------------------------------------------------------------------
# 3. SETUP BASELINE EXPLAINERS & DIMENSIONS
# ---------------------------------------------------------------------
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled.values,
    feature_names=feature_list,
    class_names=["Benign", "Web Attack"],
    mode="classification",
    random_state=42
)

shap_background_baseline = shap.sample(X_train_scaled, 100, random_state=42)
shap_explainer_xgb = shap.KernelExplainer(model=xgb_predict_proba_wrapper, data=shap_background_baseline)
shap_explainer_dnn = shap.KernelExplainer(model=dnn_predict_proba_wrapper, data=shap_background_baseline)

unique_groups_list = sorted(list(set(feature_to_group.values())))
num_groups_total = len(unique_groups_list)
group_name_to_index = {g_name: idx for idx, g_name in enumerate(unique_groups_list)}

# ---------------------------------------------------------------------
# 4. MONTE CARLO RANDOM REFERENCE SIMULATION FOR GROUP SPACE
# ---------------------------------------------------------------------
print(f"[*] Running 10,000-iteration random reference simulation over {num_groups_total} concept dimensions...")
mc_group_overlaps = {3: [], 5: [], 10: []}
all_group_pool = np.arange(num_groups_total)

for _ in range(10000):
    for k_val in [3, 5, 10]:
        rand_set_a = set(rng.choice(all_group_pool, k_val, replace=False))
        rand_set_b = set(rng.choice(all_group_pool, k_val, replace=False))
        mc_group_overlaps[k_val].append(len(rand_set_a.intersection(rand_set_b)))

expected_group_random_overlap = {k: np.mean(mc_group_overlaps[k]) for k in [3, 5, 10]}
expected_group_random_jaccard = {k: expected_group_random_overlap[k] / (2 * k - expected_group_random_overlap[k]) for k in [3, 5, 10]}

# ---------------------------------------------------------------------
# 5. MAIN EVALUATION ITERATION LOOP
# ---------------------------------------------------------------------
grouped_agreement_records = []
print("[*] Extracting attributions and aggregating weights at the group level...")

for cohort_name, indices in cohort_samples.items():
    if not indices:
        continue
    print(f"    -> Processing cohort: {cohort_name} ({len(indices)} samples)...")
    for test_idx in indices:
        instance_vector = X_test_scaled.iloc[test_idx].values
        
        # Pull standalone attributions
        lime_vector_xgb, lime_r2_xgb = extract_aligned_lime_attributions(instance_vector, xgb_predict_proba_wrapper, num_features_original)
        shap_vector_xgb = extract_aligned_shap_attributions(instance_vector, shap_explainer_xgb)
        
        lime_vector_dnn, lime_r2_dnn = extract_aligned_lime_attributions(instance_vector, dnn_predict_proba_wrapper, num_features_original)
        shap_vector_dnn = extract_aligned_shap_attributions(instance_vector, shap_explainer_dnn)
        
        for model_label, lime_v, shap_v, lime_r2 in [('XGBoost', lime_vector_xgb, shap_vector_xgb, lime_r2_xgb), 
                                                    ('DNN', lime_vector_dnn, shap_vector_dnn, lime_r2_dnn)]:
            
            lime_grouped = np.zeros(num_groups_total)
            shap_grouped = np.zeros(num_groups_total)
            
            # Map structural feature attributions to group index boundaries via absolute magnitudes
            for f_idx, f_name in enumerate(feature_list):
                g_name = feature_to_group[f_name]
                g_idx = group_name_to_index[g_name]
                lime_grouped[g_idx] += np.abs(lime_v[f_idx])
                shap_grouped[g_idx] += np.abs(shap_v[f_idx])
                
            row_metrics = {
                'Cohort': cohort_name,
                'Model': model_label,
                'Test_Index': test_idx,
                'LIME_R2': lime_r2
            }
            
            for k in [3, 5, 10]:
                top_lime_g = set(np.argsort(lime_grouped)[-k:])
                top_shap_g = set(np.argsort(shap_grouped)[-k:])
                
                overlap = len(top_lime_g.intersection(top_shap_g))
                union_len = len(top_lime_g.union(top_shap_g))
                jaccard = overlap / union_len if union_len > 0 else 0.0
                
                row_metrics[f'Overlap_{k}'] = overlap
                row_metrics[f'Jaccard_{k}'] = jaccard
                row_metrics[f'Random_Overlap_{k}'] = expected_group_random_overlap[k]
                row_metrics[f'Random_Jaccard_{k}'] = expected_group_random_jaccard[k]
                
            top_5_lime_g = set(np.argsort(lime_grouped)[-5:])
            top_5_shap_g = set(np.argsort(shap_grouped)[-5:])
            top_5_union_g = list(top_5_lime_g.union(top_5_shap_g))
            
            lime_union_slice = lime_grouped[top_5_union_g]
            shap_union_slice = shap_grouped[top_5_union_g]
            
            if np.std(lime_union_slice) == 0 or np.std(shap_union_slice) == 0:
                row_metrics['Spearman_Absolute'] = 0.0
                row_metrics['Kendall_Absolute'] = 0.0
            else:
                row_metrics['Spearman_Absolute'], _ = stats.spearmanr(lime_union_slice, shap_union_slice)
                row_metrics['Kendall_Absolute'], _ = stats.kendalltau(lime_union_slice, shap_union_slice)
                
            grouped_agreement_records.append(row_metrics)

df_grouped_agreement = pd.DataFrame(grouped_agreement_records)
print("[+] Group-level analysis calculations executed smoothly.")

# ---------------------------------------------------------------------
# 6. BOOTSTRAPPED ACADEMIC REPORT GENERATION
# ---------------------------------------------------------------------
print("[*] Performing 5,000-sample bootstrap for master confidence summary...")
grouped_report_rows = []
metrics_to_bootstrap_g = ['LIME_R2', 'Overlap_3', 'Overlap_5', 'Overlap_10', 'Jaccard_5', 'Spearman_Absolute', 'Kendall_Absolute']

unique_grouped_segments = df_grouped_agreement.groupby(['Cohort', 'Model'])

for (cohort_name, model_type), group_df in unique_grouped_segments:
    row_summary = {'Cohort': cohort_name, 'Model': model_type}
    
    for metric in metrics_to_bootstrap_g:
        mean_v, low_v, high_v = compute_bootstrapped_ci(group_df[metric], n_bootstrap=5000)
        row_summary[metric] = f"{mean_v:.3f} [{low_v:.3f}, {high_v:.3f}]"
        
    row_summary['Rand_Overlap_5'] = f"{group_df['Random_Overlap_5'].iloc[0]:.2f}"
    row_summary['Rand_Jaccard_5'] = f"{group_df['Random_Jaccard_5'].iloc[0]:.3f}"
    
    grouped_report_rows.append(row_summary)

df_grouped_master_report = pd.DataFrame(grouped_report_rows)

print("\n" + "="*145)
print("                           FINAL ACADEMIC FEATURE-GROUP LEVEL AGREEMENT REPORT WITH 95% CIs")
print("="*145)
pd.set_option('display.max_columns', None)
display(df_grouped_master_report)
print("="*145)

[*] Running 10,000-iteration random reference simulation over 39 concept dimensions...
[*] Extracting attributions and aggregating weights at the group level...
    -> Processing cohort: True Positives (50 samples)...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

[+] Group-level analysis calculations executed smoothly.
[*] Performing 5,000-sample bootstrap for master confidence summary...

                           FINAL ACADEMIC FEATURE-GROUP LEVEL AGREEMENT REPORT WITH 95% CIs


,Cohort,Model,LIME_R2,Overlap_3,Overlap_5,Overlap_10,Jaccard_5,Spearman_Absolute,Kendall_Absolute,Rand_Overlap_5,Rand_Jaccard_5
0,Borderline Cases,DNN,"0.035 [0.033, 0.037]","0.080 [0.020, 0.160]","0.400 [0.260, 0.560]","3.060 [2.800, 3.300]","0.046 [0.029, 0.064]","-0.642 [-0.667, -0.615]","-0.431 [-0.460, -0.404]",0.65,0.069
1,Borderline Cases,XGBoost,"0.043 [0.042, 0.044]","0.040 [0.000, 0.100]","0.320 [0.200, 0.460]","3.760 [3.520, 4.000]","0.036 [0.022, 0.049]","-0.613 [-0.635, -0.586]","-0.360 [-0.383, -0.336]",0.65,0.069
2,False Negatives,DNN,"0.030 [0.025, 0.035]","0.421 [0.158, 0.684]","0.842 [0.474, 1.263]","3.474 [3.053, 3.895]","0.104 [0.053, 0.165]","-0.511 [-0.635, -0.365]","-0.374 [-0.474, -0.270]",0.65,0.069
3,False Negatives,XGBoost,"0.039 [0.035, 0.042]","0.105 [0.000, 0.263]","0.579 [0.211, 1.000]","4.474 [4.158, 4.842]","0.072 [0.025, 0.129]","-0.577 [-0.623, -0.514]","-0.336 [-0.372, -0.300]",0.65,0.069
4,False Positives,DNN,"0.036 [0.034, 0.038]","0.200 [0.100, 0.320]","0.460 [0.320, 0.620]","2.940 [2.680, 3.200]","0.052 [0.036, 0.068]","-0.619 [-0.653, -0.582]","-0.415 [-0.446, -0.383]",0.65,0.069
5,False Positives,XGBoost,"0.029 [0.025, 0.033]","0.580 [0.420, 0.740]","1.520 [1.260, 1.780]","4.980 [4.660, 5.320]","0.195 [0.157, 0.234]","-0.569 [-0.617, -0.518]","-0.420 [-0.466, -0.373]",0.65,0.069
6,True Negatives,DNN,"0.013 [0.012, 0.014]","0.280 [0.140, 0.440]","0.640 [0.360, 0.960]","2.520 [2.140, 2.880]","0.086 [0.045, 0.134]","-0.587 [-0.661, -0.503]","-0.425 [-0.479, -0.365]",0.65,0.069
7,True Negatives,XGBoost,"0.021 [0.019, 0.022]","0.140 [0.040, 0.260]","0.940 [0.720, 1.160]","4.560 [4.280, 4.840]","0.113 [0.086, 0.143]","-0.634 [-0.668, -0.596]","-0.415 [-0.455, -0.377]",0.65,0.069
8,True Positives,DNN,"0.039 [0.038, 0.041]","0.080 [0.020, 0.160]","0.420 [0.280, 0.580]","2.860 [2.600, 3.140]","0.047 [0.031, 0.064]","-0.658 [-0.681, -0.632]","-0.444 [-0.469, -0.419]",0.65,0.069
9,True Positives,XGBoost,"0.045 [0.044, 0.046]","0.000 [0.000, 0.000]","0.240 [0.120, 0.360]","3.820 [3.640, 4.000]","0.027 [0.013, 0.040]","-0.637 [-0.651, -0.624]","-0.381 [-0.406, -0.359]",0.65,0.069


In [ ]:


# Construct the unique feature space using group leaders from Phase 1
reduced_features_list = []
for g_name, features in group_to_features.items():
    if "Group_" in g_name:
        reduced_features_list.append(group_representatives[g_name])
    else:
        reduced_features_list.extend(features)

reduced_features_list = sorted(list(set(reduced_features_list)))
num_features_reduced = len(reduced_features_list)

print(f"[+] halved column counts: Pruned feature space from {len(X_train_scaled.columns)} down to {num_features_reduced} targets.")

# Isolate pristine dataset arrays matching the unique concepts layout
X_train_reduced = X_train_scaled[reduced_features_list].copy()
X_val_reduced = X_val_scaled[reduced_features_list].copy()
X_test_reduced = X_test_scaled[reduced_features_list].copy()

# Sync resampled training references from Phase 0 natively 
X_train_resampled_reduced = X_train_resampled[reduced_features_list].copy()

[*] Isolating standalone independent features and group representatives...
[+] halved column counts: Pruned feature space from 78 down to 39 targets.


In [ ]:


print("[*] Re-training tree models using baseline hyperparameter configurations...")

# Extract properties directly from your global baseline object to guarantee identical training paths
xgb_hyperparams = xgb_model.get_params()
xgb_model_reduced = XGBClassifier(**xgb_hyperparams)

xgb_model_reduced.fit(
    X_train_resampled_reduced,
    y_train_resampled,
    eval_set=[(X_val_reduced, y_val)],
    verbose=False
)

# Calibrate optimal prediction probability thresholds using validation set arrays
xgb_val_probs_r = xgb_model_reduced.predict_proba(X_val_reduced)[:, 1]
p_r, r_r, thresholds_r = precision_recall_curve(y_val, xgb_val_probs_r)
f1_scores_r = (2 * p_r * r_r) / (p_r + r_r + 1e-10)
best_thresh_xgb_reduced = thresholds_r[np.argmax(f1_scores_r[:-1])]

print(f"[+] Reduced XGBoost Calibration Complete. Optimal Threshold: {best_thresh_xgb_reduced:.4f}")

[*] Re-training tree models using baseline hyperparameter configurations...
[+] Reduced XGBoost Calibration Complete. Optimal Threshold: 0.8886


In [57]:


# Structural duplicate of your baseline network modifying only input limits
class PrunedIDSDeepNeuralNetwork(nn.Module):
    def __init__(self, input_dim):
        super(PrunedIDSDeepNeuralNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1) # Yields raw logits identically
        )
    def forward(self, x):
        return self.network(x)

# Setup datasets and batch sequences matching baseline configurations
train_dataset_r = TensorDataset(
    torch.FloatTensor(X_train_resampled_reduced.values),
    torch.FloatTensor(y_train_resampled)
)
train_loader_r = DataLoader(train_dataset_r, batch_size=256, shuffle=True)

X_val_tensor_r = torch.FloatTensor(X_val_reduced.values)
y_val_tensor_r = torch.FloatTensor(y_val.values if isinstance(y_val, pd.Series) else y_val)

dnn_model_reduced = PrunedIDSDeepNeuralNetwork(input_dim=num_features_reduced)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(dnn_model_reduced.parameters(), lr=0.001)

print("[*] Retraining neural sequence over compressed inputs space...")
best_val_f1 = 0.0
best_model_state_r = None

for epoch in range(100):
    dnn_model_reduced.train()
    for batch_x, batch_y in train_loader_r:
        optimizer.zero_grad()
        outputs = dnn_model_reduced(batch_x).squeeze()
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
    dnn_model_reduced.eval()
    with torch.no_grad():
        val_logits = dnn_model_reduced(X_val_tensor_r).squeeze()
        val_probs = torch.sigmoid(val_logits).numpy()
        val_preds = (val_probs >= 0.5).astype(int)
        epoch_f1 = f1_score(y_val_tensor_r.numpy(), val_preds, zero_division=0)
        
    if epoch_f1 > best_val_f1:
        best_val_f1 = epoch_f1
        best_model_state_r = dnn_model_reduced.state_dict().copy()

if best_model_state_r is not None:
    dnn_model_reduced.load_state_dict(best_model_state_r)

# Optimize classification limits over validation split probabilities
dnn_model_reduced.eval()
with torch.no_grad():
    val_probs_cal = torch.sigmoid(dnn_model_reduced(X_val_tensor_r).squeeze()).numpy()
    
p_d_r, r_d_r, thresholds_d_r = precision_recall_curve(y_val_tensor_r.numpy(), val_probs_cal)
f1_d_r = (2 * p_d_r * r_d_r) / (p_d_r + r_d_r + 1e-10)
best_dnn_threshold_reduced = thresholds_d_r[np.argmax(f1_d_r[:-1])]

print(f"[+] Reduced DNN training complete. Optimized Threshold: {best_dnn_threshold_reduced:.4f}")

[*] Retraining neural sequence over compressed inputs space...
[+] Reduced DNN training complete. Optimized Threshold: 0.9892


In [58]:


xgb_test_probs_r = xgb_model_reduced.predict_proba(X_test_reduced)[:, 1]
xgb_test_preds_r = (xgb_test_probs_r >= best_thresh_xgb_reduced).astype(int)

dnn_model_reduced.eval()
with torch.no_grad():
    dnn_test_logits_r = dnn_model_reduced(torch.FloatTensor(X_test_reduced.values)).squeeze()
    dnn_test_probs_r = torch.sigmoid(dnn_test_logits_r).numpy()
dnn_test_preds_r = (dnn_test_probs_r >= best_dnn_threshold_reduced).astype(int)

print("\n" + "="*80)
print("             REDUCING tabular COLLINEARITY PREDICTIVE AUDIT OVER VIEW")
print("="*80)
print("[XGBoost Reduced Feature Space layout]")
print(classification_report(y_test, xgb_test_preds_r, digits=4))
print("-"*80)
print("[PyTorch DNN Reduced Feature Space layout]")
print(classification_report(y_test, dnn_test_preds_r, digits=4))
print("="*80)


             REDUCING tabular COLLINEARITY PREDICTIVE AUDIT OVER VIEW
[XGBoost Reduced Feature Space layout]
              precision    recall  f1-score   support

           0     0.9998    0.9999    0.9999     25208
           1     0.9938    0.9878    0.9908       327

    accuracy                         0.9998     25535
   macro avg     0.9968    0.9938    0.9953     25535
weighted avg     0.9998    0.9998    0.9998     25535

--------------------------------------------------------------------------------
[PyTorch DNN Reduced Feature Space layout]
              precision    recall  f1-score   support

           0     0.9991    0.9996    0.9994     25208
           1     0.9683    0.9327    0.9502       327

    accuracy                         0.9987     25535
   macro avg     0.9837    0.9662    0.9748     25535
weighted avg     0.9987    0.9987    0.9987     25535



In [59]:

print("[*] Instantiating reduced explainer frameworks over the 39-feature boundary...")

# 1. Initialize Reduced LIME
lime_explainer_reduced = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_reduced.values,
    feature_names=reduced_features_list,
    class_names=["Benign", "Web Attack"],
    mode="classification",
    random_state=42
)

# 2. Aligned Predict Proba Wrappers for Reduced Space
def xgb_predict_proba_reduced_wrapper(x_numpy):
    df_temp = pd.DataFrame(x_numpy, columns=reduced_features_list)
    return xgb_model_reduced.predict_proba(df_temp)

def dnn_predict_proba_reduced_wrapper(x_numpy):
    df_temp = pd.DataFrame(x_numpy, columns=reduced_features_list)
    dnn_model_reduced.eval()
    with torch.no_grad():
        logits = dnn_model_reduced(torch.FloatTensor(df_temp.values)).squeeze()
        probs_1 = torch.sigmoid(logits).numpy()
        if probs_1.ndim == 0:
            probs_1 = np.array([probs_1])
        probs_0 = 1.0 - probs_1
        return np.column_stack((probs_0, probs_1))

# 3. Initialize Reduced SHAP Explainers
shap_background_reduced = shap.sample(X_train_reduced, 100, random_state=42)
shap_explainer_xgb_r = shap.KernelExplainer(model=xgb_predict_proba_reduced_wrapper, data=shap_background_reduced)

# DNN always uses KernelExplainer / background sample tracking
shap_background_reduced_dnn = shap.sample(X_train_reduced, 100, random_state=42)
shap_explainer_dnn_r = shap.KernelExplainer(model=dnn_predict_proba_reduced_wrapper, data=shap_background_reduced_dnn)

print(f"[+] Reduced explainers successfully initialized. Feature dimension target: {num_features_reduced}")

[*] Instantiating reduced explainer frameworks over the 39-feature boundary...
[+] Reduced explainers successfully initialized. Feature dimension target: 39


In [60]:


reduced_agreement_records = []
print("[*] Extracting evaluations across pruned model structures...")

for cohort_name, indices in cohort_samples.items():
    if not indices:
        continue
    print(f"    -> Auditing cohort: {cohort_name} ({len(indices)} samples)...")
    for test_idx in indices:
        # Strictly isolate the 39-dimensional input vector
        instance_vector_r = X_test_reduced.iloc[test_idx].values
        
        # 1. Pull Reduced LIME Attributions
        exp_xgb_l = lime_explainer_reduced.explain_instance(
            data_row=instance_vector_r, 
            predict_fn=xgb_predict_proba_reduced_wrapper, 
            num_features=num_features_reduced
        )
        lime_v_xgb_r = np.zeros(num_features_reduced)
        for f_idx, w in exp_xgb_l.local_exp[1]: 
            lime_v_xgb_r[f_idx] = w
        
        exp_dnn_l = lime_explainer_reduced.explain_instance(
            data_row=instance_vector_r, 
            predict_fn=dnn_predict_proba_reduced_wrapper, 
            num_features=num_features_reduced
        )
        lime_v_dnn_r = np.zeros(num_features_reduced)
        for f_idx, w in exp_dnn_l.local_exp[1]: 
            lime_v_dnn_r[f_idx] = w
        
        # 2. Pull Reduced SHAP Attributions (Enforcing the Reduced Explainers)
        shap_raw_xgb = shap_explainer_xgb_r.shap_values(instance_vector_r.reshape(1, -1))
        # Safely parse list (TreeExplainer multi-class) vs array output shapes
        if isinstance(shap_raw_xgb, list):
            shap_v_xgb_r = shap_raw_xgb[1].flatten()
        elif isinstance(shap_raw_xgb, np.ndarray) and len(shap_raw_xgb.shape) == 3:
            shap_v_xgb_r = shap_raw_xgb[0, :, 1]
        else:
            shap_v_xgb_r = shap_raw_xgb.flatten()
            
        shap_raw_dnn = shap_explainer_dnn_r.shap_values(instance_vector_r.reshape(1, -1))
        if isinstance(shap_raw_dnn, list):
            shap_v_dnn_r = shap_raw_dnn[1].flatten()
        elif isinstance(shap_raw_dnn, np.ndarray) and len(shap_raw_dnn.shape) == 3:
            shap_v_dnn_r = shap_raw_dnn[0, :, 1]
        else:
            shap_v_dnn_r = shap_raw_dnn.flatten()
        
        # 3. Evaluate Consensus Metrics Side-by-Side
        for model_label, lime_vec, shap_vector, l_r2 in [('XGBoost', lime_v_xgb_r, shap_v_xgb_r, exp_xgb_l.score),
                                                        ('DNN', lime_v_dnn_r, shap_v_dnn_r, exp_dnn_l.score)]:
            
            # Hard assertion check to make sure both vectors match the reduced feature size (39)
            assert len(lime_vec) == num_features_reduced, f"LIME vector size mismatch: {len(lime_vec)}"
            assert len(shap_vector) == num_features_reduced, f"SHAP vector size mismatch: {len(shap_vector)}"
            
            row_metrics = {
                'Cohort': cohort_name,
                'Model': model_label,
                'Test_Index': test_idx,
                'LIME_R2': l_r2
            }
            
            # Compare intersections at compressed k-levels
            for k in [3, 5]:
                top_lime = set(np.argsort(np.abs(lime_vec))[-k:])
                top_shap = set(np.argsort(np.abs(shap_vector))[-k:])
                
                overlap = len(top_lime.intersection(top_shap))
                jaccard = overlap / len(top_lime.union(top_shap)) if len(top_lime.union(top_shap)) > 0 else 0.0
                
                row_metrics[f'Overlap_{k}'] = overlap
                row_metrics[f'Jaccard_{k}'] = jaccard
                
            # Localized rank correlation calculated over the top 5 union
            top_5_lime = set(np.argsort(np.abs(lime_vec))[-5:])
            top_5_shap = set(np.argsort(np.abs(shap_vector))[-5:])
            top_5_union = list(top_5_lime.union(top_5_shap))
            
            lime_union_slice = np.abs(lime_vec)[top_5_union]
            shap_union_slice = np.abs(shap_vector)[top_5_union]
            
            if np.std(lime_union_slice) == 0 or np.std(shap_union_slice) == 0:
                row_metrics['Spearman_Absolute'] = 0.0
            else:
                row_metrics['Spearman_Absolute'], _ = stats.spearmanr(lime_union_slice, shap_union_slice)
                
            reduced_agreement_records.append(row_metrics)

df_reduced_agreement = pd.DataFrame(reduced_agreement_records)
print("[+] Reduced space evaluation loops executed flawlessly with strict shape verification!")

[*] Extracting evaluations across pruned model structures...
    -> Auditing cohort: True Positives (50 samples)...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

[+] Reduced space evaluation loops executed flawlessly with strict shape verification!


In [61]:

reduced_report_rows = []

for (cohort_name, model_type), group_df in df_reduced_agreement.groupby(['Cohort', 'Model']):
    row_summary = {'Cohort': cohort_name, 'Model': model_type}
    for metric in ['LIME_R2', 'Overlap_3', 'Overlap_5', 'Jaccard_5', 'Spearman_Absolute']:
        mean_v, low_v, high_v = compute_bootstrapped_ci(group_df[metric], n_bootstrap=5000)
        row_summary[metric] = f"{mean_v:.3f} [{low_v:.3f}, {high_v:.3f}]"
    reduced_report_rows.append(row_summary)

print("\n" + "="*145)
print("                   FINAL MASTER REPORT: EXPLAINER AGREEMENT OVER PRUNED NON-COLLINEAR TRAINED MODELS")
print("="*145)
display(pd.DataFrame(reduced_report_rows))
print("="*145)


                   FINAL MASTER REPORT: EXPLAINER AGREEMENT OVER PRUNED NON-COLLINEAR TRAINED MODELS


,Cohort,Model,LIME_R2,Overlap_3,Overlap_5,Jaccard_5,Spearman_Absolute
0,Borderline Cases,DNN,"0.016 [0.015, 0.016]","1.000 [0.860, 1.140]","1.760 [1.560, 1.960]","0.224 [0.193, 0.256]","-0.374 [-0.435, -0.313]"
1,Borderline Cases,XGBoost,"0.016 [0.016, 0.017]","1.280 [1.140, 1.420]","2.240 [2.020, 2.460]","0.302 [0.265, 0.340]","-0.194 [-0.258, -0.128]"
2,False Negatives,DNN,"0.015 [0.014, 0.016]","0.895 [0.684, 1.105]","1.474 [1.105, 1.895]","0.186 [0.132, 0.247]","-0.246 [-0.367, -0.125]"
3,False Negatives,XGBoost,"0.016 [0.015, 0.017]","1.263 [0.947, 1.579]","2.842 [2.421, 3.263]","0.420 [0.341, 0.497]","-0.105 [-0.254, 0.025]"
4,False Positives,DNN,"0.015 [0.014, 0.016]","0.900 [0.740, 1.060]","1.440 [1.220, 1.660]","0.179 [0.148, 0.210]","-0.425 [-0.500, -0.346]"
5,False Positives,XGBoost,"0.015 [0.013, 0.018]","1.040 [0.860, 1.220]","1.940 [1.700, 2.180]","0.256 [0.218, 0.297]","-0.369 [-0.438, -0.303]"
6,True Negatives,DNN,"0.005 [0.005, 0.005]","0.300 [0.160, 0.480]","0.900 [0.700, 1.120]","0.107 [0.081, 0.134]","-0.621 [-0.688, -0.547]"
7,True Negatives,XGBoost,"0.010 [0.009, 0.011]","0.920 [0.680, 1.180]","1.840 [1.560, 2.100]","0.243 [0.202, 0.285]","-0.310 [-0.437, -0.175]"
8,True Positives,DNN,"0.017 [0.016, 0.017]","1.040 [0.900, 1.180]","1.600 [1.440, 1.780]","0.198 [0.172, 0.227]","-0.363 [-0.415, -0.310]"
9,True Positives,XGBoost,"0.017 [0.017, 0.018]","1.500 [1.340, 1.640]","2.300 [2.080, 2.500]","0.311 [0.276, 0.348]","-0.135 [-0.194, -0.075]"
